In [1]:
import pandas as pd
import random
from cryptography.fernet import Fernet
import hashlib
import re

# Load the customer dataset from the CSV file in the 'data' folder
df = pd.read_csv('customer_data.csv')

# Get the number of rows in the dataset
num_rows = len(df)

In [2]:
# Task 1: Anonymize the names column and prevent duplication
name_mapping = {}
def anonymize_names(row):
    name = row['Name']   
    # Check if the name was already anonymized
    if name in name_mapping:
        return name_mapping[name]   
    # Generate a unique hash-based identifier
    hashed_name = hashlib.sha256(name.encode()).hexdigest()[:10]  # Shorten for readability  
    # Store mapping
    name_mapping[name] = hashed_name
    
    return hashed_name

# Uncomment the line below to apply the anonymize_names function to the 'Name' column
df['Name'] = df.apply(anonymize_names, axis=1)

df.to_csv('anonymized_customer_data.csv', index=False)

print("Anonymization complete. Saved to anonymized_customer_data.csv")

Anonymization complete. Saved to anonymized_customer_data.csv


In [3]:
# Task 2: Encrypt the phone numbers column using AES
# Use the provided AES key
aes_key = b'mZoTMS5zmUYjAdLbM1GRv3z5fGaz5B4IXi693pX0514='
cipher = Fernet(aes_key)
# AES hint: You need to create a Fernet cipher object using the AES key.
# Then, encrypt each phone number using the cipher object.

# Uncomment the lines below to apply the encryption to the 'Phone' column
def encrypt_phone_numbers(phone):
    phone = str(phone)  # Convert to string to ensure proper encryption
    # Encrypt phone number using AES
    encrypted_phone = cipher.encrypt(phone.encode()).decode()
    
    return encrypted_phone


df['Phone'] = df['Phone'].apply(encrypt_phone_numbers)

df.to_csv('encrypted_customer_data.csv', index=False)

print("Encryption complete. Saved to encrypted_customer_data.csv")


Encryption complete. Saved to encrypted_customer_data.csv


In [ ]:
# Task 3: Leave the email address as-is.

In [4]:
# Task 4: Encrypt the credit card numbers column using SHA-256 hashing

# SHA hint: You need to create a SHA-256 object, update it with the credit card number,
# and then retrieve the hexadecimal representation of the hash.

# Uncomment the lines below to apply the encryption to the 'Credit Card Number' column
def encrypt_credit_card_numbers(card):
    # Convert the credit card number to string (ensure compatibility)
    card_str = str(card)
    # Create a SHA-256 hash object
    sha = hashlib.sha256()
    # Update the hash object with the credit card number
    sha.update(card_str.encode())
    # Return the hexadecimal representation of the hash
    return sha.hexdigest()


df['Credit Card Number'] = df['Credit Card Number'].apply(encrypt_credit_card_numbers)

df.to_csv('hashed_customer_data.csv', index=False)

In [5]:
# Task 5: Anonymize the address column
def anonymize_addresses(address):
    # Default values for City, State, and ZIP Code
    city = ''
    state = ''
    zip_code = ''
    
    try:
        # Extract ZIP Code (5-digit number at the end of the address)
        zip_match = re.search(r'\b\d{5}\b', address)
        if zip_match:
            zip_code = zip_match.group()
        
        # Extract State (2-letter code before ZIP)
        state_match = re.search(r'\b[A-Z]{2}\b(?=\s+\d{5})', address)
        if state_match:
            state = state_match.group()
        
        # Handle military addresses
        if 'DPO' in address or 'FPO' in address or 'APO' in address or 'PSC' in address:
            # Use specific military keywords as the City placeholder
            if 'PSC' in address:
                city = 'PSC'
            elif 'DPO' in address:
                city = 'DPO'
            elif 'FPO' in address:
                city = 'FPO'
            elif 'APO' in address:
                city = 'APO'
        else:
            # Handle standard addresses
            # Extract City (assume it's the part before the state, excluding street and suite information)
            city_match = re.search(r'(?:.*?,)?\s*(.*?)(?=,\s*' + state + r')', address) if state else None
            if city_match:
                city = city_match.group().strip()
                
                # Remove street numbers and suite/apt details from City
                city = re.sub(r'\d.*?(,|$)', '', city).strip()
        
        # Fallback handling for unrecognized formats
        if not city or not state or not zip_code:
            print(f"Unrecognized address format: {address}")
    
    except Exception as e:
        print(f"Error parsing address: {address}. Exception: {e}")
    
    return city, state, zip_code

# Apply the anonymization function while keeping your original method
df['Address'] = df['Address'].apply(anonymize_addresses)

# Split the 'Address' lists into new columns
df[['City', 'State', 'ZIP Code']] = pd.DataFrame(df['Address'].tolist(), index=df.index)

# Remove the original 'Address' column
df.drop(columns=['Address'], inplace=True)

# Save the modified dataset
df.to_csv('anonymized_customer_data.csv', index=False)

print("Address anonymization and splitting complete. Saved to anonymized_customer_data.csv")


Address anonymization and splitting complete. Saved to anonymized_customer_data.csv


In [6]:
# Task 6: Remove the "Customer Notes" field

df.drop(columns=['Customer Notes'], inplace=True)
# Save the updated dataset
df.to_csv('updated_customer_data.csv', index=False)

print("Customer Notes field removed. Saved to updated_customer_data.csv")

Customer Notes field removed. Saved to updated_customer_data.csv


In [7]:
# Task 7: Save the anonymized data to a new CSV file

df.to_csv('anonymized_customer_data.csv', index=False)

In [ ]:
# Task 8: Write a report (1-2 pages) on the importance of data governance
# - Explain how data anonymization and encryption contribute to data governance and data privacy